In [ ]:
import mysql.connector
from mysql.connector import Error

# Database config
config = {
    "host": "127.0.0.1",
    "port": 3306,
    "user": "db_user",
    "password": "6equj5_db_user",
    "database": "home_db"
}

with open("create_tables.sql", "r") as f:
    sql_script = f.read()

try:
    connection = mysql.connector.connect(**config)
    cursor = connection.cursor()
    for statement in sql_script.split(';'):
        stmt = statement.strip()
        if stmt and not stmt.startswith('--') and not stmt.upper().startswith('USE'):
            try:
                cursor.execute(stmt)
                print(f"Executed: {stmt[:50]}...")
            except Error as e:
                print(f" Error in statement: {stmt[:50]}... → {e}")

    connection.commit()
    print("All tables created successfully!")

except Error as e:
    print(f"Database error: {e}")

finally:
    if 'cursor' in locals():
        cursor.close()
    if 'connection' in locals() and connection.is_connected():
        connection.close()
        print("🔒 Connection closed.")

In [46]:
import pandas as pd
import json
from sqlalchemy import create_engine

mysql_user = "db_user"
mysql_pass = "6equj5_db_user"
mysql_host = "127.0.0.1"
mysql_port = "3306"
mysql_db = "home_db"

engine = create_engine(f"mysql+pymysql://{mysql_user}:{mysql_pass}@{mysql_host}:{mysql_port}/{mysql_db}")

json_file = "data/fake_property_data_new.json"
field_config_file = "data/Field Config.xlsx"
with open(json_file, "r") as f:
    data = json.load(f)

field_config = pd.read_excel(field_config_file)
flat_column_to_table = {}
for _, row in field_config.iterrows():
    col = row['Column Name']
    tbl = row['Target Table']
    if isinstance(tbl, str) and tbl.lower() not in {'valuation', 'rehab', 'hoa', 'taxes'}:
        flat_column_to_table[col] = tbl

tables = {
    'property': [],
    'leads': [],
    'valuation': [],
    'rehab': [],
    'hoa': [],
    'taxes': []
}

def flatten_value(val):
    if isinstance(val, (dict, list)):
        return json.dumps(val)
    return val

for record in data:
    property_row = {}
    leads_row = {}

    for col, table in flat_column_to_table.items():
        value = record.get(col)
        if table.lower() == "property":
            property_row[col] = flatten_value(value)
        elif table.lower() == "leads":
            leads_row[col] = flatten_value(value)

    tables['property'].append(property_row)
    if leads_row:
        tables['leads'].append(leads_row)

    property_id = property_row.get('property_id') or record.get('property_id')

    # -Valuation 
    val_data = record.get("Valuation")
    if isinstance(val_data, list):
        for item in val_data: 
            if isinstance(item, dict):
                row = {k: flatten_value(v) for k, v in item.items()}
                if property_id is not None:
                    row['property_id'] = property_id
                tables['valuation'].append(row)

    # -Rehab
    rehab_data = record.get("Rehab")
    if isinstance(rehab_data, list):
        for item in rehab_data:  # ✅ Fixed
            if isinstance(item, dict):
                row = {k: flatten_value(v) for k, v in item.items()}
                if property_id is not None:
                    row['property_id'] = property_id
                tables['rehab'].append(row)

    # -HOA
    hoa_data = record.get("HOA")
    if isinstance(hoa_data, list):
        for item in hoa_data:  # ✅ Fixed
            if isinstance(item, dict):
                row = {k: flatten_value(v) for k, v in item.items()}
                if property_id is not None:
                    row['property_id'] = property_id
                tables['hoa'].append(row)

    # -Taxes 
    taxes_raw = record.get("Taxes") or record.get("taxes")
    if taxes_raw is None:
        pass
    elif isinstance(taxes_raw, list):
        for item in taxes_raw:  
            row = {}
            if isinstance(item, dict):
                row['Taxes'] = flatten_value(item.get('Taxes'))
            elif isinstance(item, (int, float)):
                row['Taxes'] = item
            else:
                continue
            if property_id is not None:
                row['property_id'] = property_id
            tables['taxes'].append(row)
    elif isinstance(taxes_raw, (int, float)):
        tables['taxes'].append({'Taxes': taxes_raw, 'property_id': property_id})

def filter_columns(df, table_name):
    try:
        existing_cols = pd.read_sql(f"SELECT * FROM {table_name} LIMIT 0", engine).columns.tolist()
        valid_cols = [col for col in df.columns if col in existing_cols]
        return df[valid_cols]
    except Exception as e:
        print(f"Warning: Could not validate columns for '{table_name}': {e}")
        return df

for table_name, rows in tables.items():
    if not rows:
        print(f"⏭Skipping empty table '{table_name}'")
        continue

    df = pd.DataFrame(rows)
    df = filter_columns(df, table_name)

    if df.empty:
        print(f"No valid columns to insert into '{table_name}'")
        continue

    try:
        df.to_sql(
            name=table_name,
            con=engine,
            if_exists='append',
            index=False,
            method='multi'
        )
        print(f"Loaded {len(df)} rows into '{table_name}'")
    except Exception as e:
        print(f" Failed to insert into '{table_name}': {e}")

print("All data loaded successfully!")

Loaded 10088 rows into 'property'
Loaded 10088 rows into 'leads'
Loaded 24898 rows into 'valuation'
Loaded 20219 rows into 'rehab'
Loaded 10100 rows into 'hoa'
Loaded 10088 rows into 'taxes'
All data loaded successfully!
